### I found that bulk is not really necessary and also removing bulk, low demand and zero demand products might improve the model fot the final model.

## This might be the last modelling starting with creating a dataset for that.

In [2]:
from __future__ import annotations

import hashlib
import json
import os
import shutil
import stat
import uuid
from datetime import datetime, timezone
from pathlib import Path
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd

# =============================================================================
# EDEN NORMAL-DEMAND MODEL V2
# ND01 — CREATE ORGANISED WORKSPACE AND COPY CANONICAL SOURCE DATASET
# =============================================================================

PROJECT_ROOT = Path("/Users/ryansmac/Desktop/Meng Project")
EDEN_ROOT = PROJECT_ROOT / "eden_datasets"
SOURCE_FILENAME = "UL_EDEN_canonical_product_daily_demand_forecasting_final.csv"

MODEL_FOLDER_NAME = "eden_normal_demand_model_v2"
MODEL_ROOT = EDEN_ROOT / MODEL_FOLDER_NAME

MEMORY_ROOT = MODEL_ROOT / "00_project_memory"
DATASET_ROOT = MODEL_ROOT / "01_datasets"
SOURCE_COPY_ROOT = DATASET_ROOT / "00_source_copy"
INTERMEDIATE_DATA_ROOT = DATASET_ROOT / "01_intermediate"
FINAL_DATA_ROOT = DATASET_ROOT / "02_final"
FEATURE_ROOT = MODEL_ROOT / "02_feature_engineering"
MODEL_CANDIDATE_ROOT = MODEL_ROOT / "03_models" / "00_candidates"
MODEL_FINAL_ROOT = MODEL_ROOT / "03_models" / "01_final"
PREDICTION_ROOT = MODEL_ROOT / "04_predictions"
VALIDATION_ROOT = MODEL_ROOT / "05_validation"
REPORT_ROOT = MODEL_ROOT / "06_reports"
NOTEBOOK_ROOT = MODEL_ROOT / "07_notebooks"
CHECKPOINT_ROOT = MODEL_ROOT / "08_checkpoints"
LOG_ROOT = MODEL_ROOT / "09_logs"

SOURCE_COPY_PATH = SOURCE_COPY_ROOT / SOURCE_FILENAME
SOURCE_SNAPSHOT_PATH = SOURCE_COPY_ROOT / "ND01_source_snapshot.json"
SOURCE_README_PATH = SOURCE_COPY_ROOT / "README.md"

README_PATH = MODEL_ROOT / "README.md"
AGENTS_PATH = MODEL_ROOT / "AGENTS.md"
GITIGNORE_PATH = MODEL_ROOT / ".gitignore"

PROJECT_CONTEXT_PATH = MEMORY_ROOT / "PROJECT_CONTEXT.md"
WORKFLOW_PATH = MEMORY_ROOT / "WORKFLOW.md"
DECISIONS_PATH = MEMORY_ROOT / "DECISIONS.md"
FILES_AND_PATHS_PATH = MEMORY_ROOT / "FILES_AND_PATHS.md"
METRICS_AND_RESULTS_PATH = MEMORY_ROOT / "METRICS_AND_RESULTS.md"
CURRENT_HANDOFF_PATH = MEMORY_ROOT / "CURRENT_HANDOFF.md"
CHAT_INDEX_PATH = MEMORY_ROOT / "CHAT_INDEX.md"
DATA_DICTIONARY_PATH = MEMORY_ROOT / "SOURCE_DATA_DICTIONARY.md"

VALIDATION_PATH = VALIDATION_ROOT / "ND01_source_copy_validation.csv"
MANIFEST_PATH = VALIDATION_ROOT / "ND01_artifact_hash_manifest.csv"
CHECKPOINT_PATH = CHECKPOINT_ROOT / "ND01_checkpoint.json"
CHECKPOINT_SHA_PATH = CHECKPOINT_ROOT / "ND01_checkpoint.sha256"
LOCK_PATH = CHECKPOINT_ROOT / "ND01_workspace_and_source_copy_lock.json"
LOCK_SHA_PATH = CHECKPOINT_ROOT / "ND01_workspace_and_source_copy_lock.sha256"
LOG_PATH = LOG_ROOT / "ND01_workspace_setup_log.txt"

STEP_ID = "ND01"
STATUS = "ND01_SOURCE_WORKSPACE_CREATED_READY_FOR_ND02"
NOW_UTC = datetime.now(timezone.utc)
NOW_LOCAL = NOW_UTC.astimezone(ZoneInfo("Europe/Dublin"))


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def write_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")


def write_json(path: Path, payload: dict) -> None:
    write_text(path, json.dumps(payload, indent=2, ensure_ascii=False) + "\n")


def write_csv(path: Path, frame: pd.DataFrame) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False)


def make_read_only(path: Path) -> None:
    if path.is_file():
        path.chmod(stat.S_IRUSR | stat.S_IRGRP | stat.S_IROTH)


def locate_source_dataset() -> Path:
    expected = EDEN_ROOT / SOURCE_FILENAME
    if expected.is_file():
        return expected

    matches = [
        path
        for path in EDEN_ROOT.rglob(SOURCE_FILENAME)
        if MODEL_FOLDER_NAME not in path.parts
    ]

    if len(matches) == 1:
        return matches[0]
    if not matches:
        raise FileNotFoundError(
            "The canonical source dataset was not found.\n"
            f"Expected location:\n{expected}"
        )
    raise RuntimeError(
        "More than one matching canonical source dataset was found.\n"
        + "\n".join(f"- {path}" for path in matches)
    )


# =============================================================================
# PRE-FLIGHT SAFETY
# =============================================================================

if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(f"Project root does not exist:\n{PROJECT_ROOT}")
if not EDEN_ROOT.is_dir():
    raise FileNotFoundError(f"Eden dataset root does not exist:\n{EDEN_ROOT}")

SOURCE_PATH = locate_source_dataset()

if MODEL_ROOT.exists():
    raise FileExistsError(
        "The modelling workspace already exists, so ND01 will not overwrite it.\n"
        f"Existing path:\n{MODEL_ROOT}"
    )

for old_stage in EDEN_ROOT.glob(f".{MODEL_FOLDER_NAME}_staging_*"):
    if old_stage.is_dir():
        shutil.rmtree(old_stage)

STAGING_ROOT = EDEN_ROOT / f".{MODEL_FOLDER_NAME}_staging_{uuid.uuid4().hex}"
STAGING_ROOT.mkdir(parents=True, exist_ok=False)

relative_directories = [
    Path("00_project_memory"),
    Path("01_datasets/00_source_copy"),
    Path("01_datasets/01_intermediate"),
    Path("01_datasets/02_final"),
    Path("02_feature_engineering"),
    Path("03_models/00_candidates"),
    Path("03_models/01_final"),
    Path("04_predictions"),
    Path("05_validation"),
    Path("06_reports"),
    Path("07_notebooks"),
    Path("08_checkpoints"),
    Path("09_logs"),
]

try:
    for relative_directory in relative_directories:
        (STAGING_ROOT / relative_directory).mkdir(parents=True, exist_ok=True)

    staged_source_copy = STAGING_ROOT / SOURCE_COPY_PATH.relative_to(MODEL_ROOT)
    shutil.copy2(SOURCE_PATH, staged_source_copy)

    source_sha256 = sha256_file(SOURCE_PATH)
    copied_sha256 = sha256_file(staged_source_copy)
    if source_sha256 != copied_sha256:
        raise AssertionError("The copied source does not match the original source.")

    # =========================================================================
    # SOURCE VALIDATION
    # =========================================================================

    source_df = pd.read_csv(staged_source_copy, low_memory=False)

    required_columns = {
        "Date",
        "CanonicalProductID",
        "NormalDemand",
        "BulkDemand",
        "TotalDemand",
    }
    missing_columns = sorted(required_columns - set(source_df.columns))
    if missing_columns:
        raise AssertionError(
            "The source dataset is missing required columns:\n"
            + "\n".join(f"- {column}" for column in missing_columns)
        )

    source_df["Date"] = pd.to_datetime(source_df["Date"], errors="raise")
    source_df["CanonicalProductID"] = source_df["CanonicalProductID"].astype(str)

    demand_columns = ["NormalDemand", "BulkDemand", "TotalDemand"]
    for column in demand_columns:
        source_df[column] = pd.to_numeric(source_df[column], errors="raise").astype(float)

    demand_array = source_df[demand_columns].to_numpy(dtype=float)
    non_finite_demand_values = int((~np.isfinite(demand_array)).sum())
    negative_demand_rows = int((source_df[demand_columns] < 0).any(axis=1).sum())
    duplicate_product_date_rows = int(
        source_df.duplicated(["Date", "CanonicalProductID"]).sum()
    )

    reconciliation_difference = (
        source_df["NormalDemand"]
        + source_df["BulkDemand"]
        - source_df["TotalDemand"]
    )
    maximum_reconciliation_difference = float(reconciliation_difference.abs().max())

    if non_finite_demand_values != 0:
        raise AssertionError("Non-finite demand values were found.")
    if negative_demand_rows != 0:
        raise AssertionError("Negative demand rows were found.")
    if duplicate_product_date_rows != 0:
        raise AssertionError("Duplicate product-date rows were found.")
    if maximum_reconciliation_difference > 1e-9:
        raise AssertionError(
            "NormalDemand + BulkDemand does not reconcile to TotalDemand."
        )

    row_count = int(len(source_df))
    column_count = int(len(source_df.columns))
    product_count = int(source_df["CanonicalProductID"].nunique())
    operating_date_count = int(source_df["Date"].nunique())
    minimum_date = source_df["Date"].min().date().isoformat()
    maximum_date = source_df["Date"].max().date().isoformat()

    normal_demand_total = float(source_df["NormalDemand"].sum())
    bulk_demand_total = float(source_df["BulkDemand"].sum())
    total_demand_total = float(source_df["TotalDemand"].sum())
    bulk_share_percentage = (
        100.0 * bulk_demand_total / total_demand_total
        if total_demand_total != 0
        else np.nan
    )

    source_snapshot = {
        "StepID": STEP_ID,
        "Status": STATUS,
        "CreatedUTC": NOW_UTC.isoformat(),
        "CreatedLocal": NOW_LOCAL.isoformat(),
        "OriginalSourcePath": str(SOURCE_PATH),
        "CopiedSourcePath": str(SOURCE_COPY_PATH),
        "OriginalSourceSHA256": source_sha256,
        "CopiedSourceSHA256": copied_sha256,
        "CopyIsByteIdentical": True,
        "SourceBytes": int(SOURCE_PATH.stat().st_size),
        "Rows": row_count,
        "Columns": column_count,
        "Products": product_count,
        "OperatingDates": operating_date_count,
        "MinimumDate": minimum_date,
        "MaximumDate": maximum_date,
        "NormalDemandTotal": normal_demand_total,
        "BulkDemandTotal": bulk_demand_total,
        "TotalDemandTotal": total_demand_total,
        "BulkSharePercentage": bulk_share_percentage,
        "MaximumDemandReconciliationDifference": maximum_reconciliation_difference,
        "DuplicateProductDateRows": duplicate_product_date_rows,
        "NegativeDemandRows": negative_demand_rows,
        "NonFiniteDemandValues": non_finite_demand_values,
        "SourceModifiedDuringStep": False,
        "CopiedDatasetModifiedDuringStep": False,
    }

    staged_source_snapshot = STAGING_ROOT / SOURCE_SNAPSHOT_PATH.relative_to(MODEL_ROOT)
    write_json(staged_source_snapshot, source_snapshot)

    validation = pd.DataFrame(
        [
            {
                "Check": "Original source exists",
                "Expected": True,
                "Actual": SOURCE_PATH.is_file(),
                "Passed": SOURCE_PATH.is_file(),
            },
            {
                "Check": "Copied source exists",
                "Expected": True,
                "Actual": staged_source_copy.is_file(),
                "Passed": staged_source_copy.is_file(),
            },
            {
                "Check": "Source and copied hashes match",
                "Expected": source_sha256,
                "Actual": copied_sha256,
                "Passed": source_sha256 == copied_sha256,
            },
            {
                "Check": "Required demand columns present",
                "Expected": True,
                "Actual": len(missing_columns) == 0,
                "Passed": len(missing_columns) == 0,
            },
            {
                "Check": "Duplicate product-date rows",
                "Expected": 0,
                "Actual": duplicate_product_date_rows,
                "Passed": duplicate_product_date_rows == 0,
            },
            {
                "Check": "Negative demand rows",
                "Expected": 0,
                "Actual": negative_demand_rows,
                "Passed": negative_demand_rows == 0,
            },
            {
                "Check": "Non-finite demand values",
                "Expected": 0,
                "Actual": non_finite_demand_values,
                "Passed": non_finite_demand_values == 0,
            },
            {
                "Check": "Maximum demand reconciliation difference",
                "Expected": "<= 1e-9",
                "Actual": maximum_reconciliation_difference,
                "Passed": maximum_reconciliation_difference <= 1e-9,
            },
            {
                "Check": "Original source modified",
                "Expected": False,
                "Actual": False,
                "Passed": True,
            },
            {
                "Check": "Dataset transformed during ND01",
                "Expected": False,
                "Actual": False,
                "Passed": True,
            },
        ]
    )

    if not validation["Passed"].all():
        raise AssertionError(
            "ND01 validation failed:\n"
            + validation.loc[~validation["Passed"]].to_string(index=False)
        )

    staged_validation = STAGING_ROOT / VALIDATION_PATH.relative_to(MODEL_ROOT)
    write_csv(staged_validation, validation)

    # =========================================================================
    # DOCUMENTATION
    # No Markdown triple-backtick fences are used inside these Python strings.
    # =========================================================================

    folder_tree = "\n".join(
        [
            f"{MODEL_FOLDER_NAME}/",
            "  AGENTS.md",
            "  README.md",
            "  00_project_memory/",
            "  01_datasets/",
            "    00_source_copy/",
            "    01_intermediate/",
            "    02_final/",
            "  02_feature_engineering/",
            "  03_models/",
            "    00_candidates/",
            "    01_final/",
            "  04_predictions/",
            "  05_validation/",
            "  06_reports/",
            "  07_notebooks/",
            "  08_checkpoints/",
            "  09_logs/",
        ]
    )

    readme_text = f"""# Eden Normal-Demand Model V2

## Purpose

This workspace contains the revised Eden Restaurant forecasting workflow.
The revised model will forecast ordinary normal product demand. Confirmed bulk
orders will be handled as known external operational inputs.

## Operational quantity rule

Required quantity = forecast normal demand + confirmed bulk-order quantity

## Current status

- Step completed: {STEP_ID}
- Status: {STATUS}
- Canonical source copied unchanged: {SOURCE_COPY_PATH}
- No feature engineering has been performed.
- No products have been filtered.
- No model has been fitted.
- No previous Eden artefact has been overwritten.

## Modelling principles

1. Use NormalDemand as the revised model target.
2. Retain BulkDemand for audit and reconciliation only.
3. Rebuild lag and rolling predictors using past NormalDemand only.
4. Determine product eligibility using information available before each forecast.
5. Focus the main model on established high- and moderate-demand products.
6. Retain low-demand and cold-start products for fallback and audit.
7. Report forecasting error together with demand coverage.
8. Use chronological, leakage-safe evaluation.
9. Preserve the original locked Eden models as benchmarks.
10. Do not describe 100 minus WAPE as conventional accuracy.

## Folder structure

{folder_tree}

## Next step

ND02 will create the leakage-safe normal-demand panel, prior-only product
eligibility fields, demand coverage summaries and fallback routes.
"""

    agents_text = f"""# AGENTS.md

## Project

Eden Restaurant normal-demand forecasting model, version 2.

## Authoritative status

- Completed step: {STEP_ID}
- Status: {STATUS}
- Next step: ND02
- Workspace root: {MODEL_ROOT}
- Immutable source copy: {SOURCE_COPY_PATH}

## Objective

Create a product-level daily forecasting system for ordinary restaurant demand.
Bulk orders are treated as confirmed external quantities rather than uncertain
routine demand that must be inferred from point-of-sale history.

## Non-negotiable rules

1. The revised target is NormalDemand.
2. Do not train the revised model on TotalDemand.
3. Do not create revised historical predictors from TotalDemand.
4. All lags, rolling statistics and expanding statistics must use prior NormalDemand.
5. Do not classify products using complete-period or future demand.
6. Product rank, segment and eligibility must be recalculated from prior data only.
7. Low-demand and cold-start products must remain in the data and use an explicit fallback.
8. Report WAPE together with the percentage of normal demand covered.
9. Bias is prediction minus actual.
10. Do not round forecasts before scoring.
11. Use chronological validation only.
12. March 2026 has already been examined and is not a new untouched final test.
13. Never overwrite an existing lock, checkpoint, source snapshot or final model.
14. Never modify the CSV in 01_datasets/00_source_copy.
15. Do not commit raw or derived Eden datasets to Git.

## Planned routing

Established high/moderate products -> main normal-demand model
Low-demand/zero-heavy/insufficient-history products -> explicit fallback
Confirmed bulk orders -> external quantity added after prediction

## Safe working procedure

1. Read 00_project_memory/CURRENT_HANDOFF.md.
2. Verify the previous lock and checkpoint.
3. Write new artefacts into staging.
4. Validate all outputs.
5. Calculate SHA-256 hashes.
6. Commit atomically.
7. Create a new immutable lock.
"""

    source_readme_text = f"""# Canonical Source Copy

This directory contains the immutable source snapshot for the revised
normal-demand modelling workflow.

Original file: {SOURCE_PATH}
Copied file: {SOURCE_COPY_PATH}
Original SHA-256: {source_sha256}
Copied SHA-256: {copied_sha256}
Byte-identical copy: True

Rules:
- Do not edit this CSV.
- Do not add engineered features to this CSV.
- Do not remove bulk rows from this CSV.
- Do not remove low-demand products from this CSV.
- Write transformations to 01_datasets/01_intermediate.
- Write final prepared datasets to 01_datasets/02_final.
"""

    project_context_text = f"""# Project Context

## Objective

Develop an operational forecasting prototype for Eden Restaurant that predicts
product demand and supports food-waste prevention.

## Revised modelling objective

Forecast NormalDemand rather than TotalDemand. Confirmed bulk orders will be
added externally to the predicted normal demand.

## Source profile

- Rows: {row_count:,}
- Columns: {column_count:,}
- Products: {product_count:,}
- Operating dates: {operating_date_count:,}
- Date range: {minimum_date} to {maximum_date}
- Normal demand: {normal_demand_total:,.6f}
- Bulk demand: {bulk_demand_total:,.6f}
- Total demand: {total_demand_total:,.6f}
- Bulk share: {bulk_share_percentage:.6f}%

## Current status

The workspace and immutable source snapshot are complete. No modelling
transformation has yet been applied.
"""

    workflow_text = """# Workflow

## Completed

### ND01 — Workspace and source snapshot

- Created the complete modelling workspace.
- Copied the canonical product-day dataset unchanged.
- Verified source integrity and demand reconciliation.
- Created agent documentation, project memory, validation, hashes and lock.

## Planned

### ND02 — Normal-demand panel and product eligibility
- Preserve all source rows.
- Set NormalDemand as the modelling target.
- Create prior-only product history and ranking fields.
- Define main-model and fallback routes.
- Quantify demand coverage.

### ND03 — Normal-demand feature engineering
- Rebuild lag and rolling features from NormalDemand.
- Produce chronological model-selection datasets.
- Complete leakage and missingness audits.

### ND04 — Baselines and fallback evaluation

### ND05 — Candidate daily models

### ND06 — Tuning, calibration and final selection

### ND07 — Daily-to-weekly aggregation and rolling updates

### ND08 — Arbitrary future-date inference pipeline

### ND09 — Untouched future evaluation
"""

    decisions_text = """# Decisions

## ND01 locked decisions

1. The canonical product-day dataset is the source of truth.
2. The revised target is NormalDemand.
3. BulkDemand remains in source and audit data but is excluded from the statistical target.
4. Confirmed bulk quantities will be added externally.
5. Low-demand and zero-heavy products will not be deleted from the source.
6. Main-model eligibility will be calculated dynamically from prior data.
7. Insufficient-history and outside-scope products will use an explicit fallback.
8. The original locked daily model remains the benchmark.
9. All model evaluation must be chronological and leakage-safe.
10. Accuracy results must be accompanied by demand coverage.
"""

    files_and_paths_text = f"""# Files and Paths

Workspace root: {MODEL_ROOT}
Original canonical source: {SOURCE_PATH}
Immutable copied source: {SOURCE_COPY_PATH}
Source snapshot: {SOURCE_SNAPSHOT_PATH}
Validation: {VALIDATION_PATH}
Hash manifest: {MANIFEST_PATH}
Checkpoint: {CHECKPOINT_PATH}
Lock: {LOCK_PATH}
Intermediate datasets: {INTERMEDIATE_DATA_ROOT}
Final datasets: {FINAL_DATA_ROOT}
Feature artefacts: {FEATURE_ROOT}
"""

    metrics_text = f"""# Metrics and Results

## ND01 source profile

- Rows: {row_count:,}
- Columns: {column_count:,}
- Products: {product_count:,}
- Operating dates: {operating_date_count:,}
- Minimum date: {minimum_date}
- Maximum date: {maximum_date}
- Normal demand total: {normal_demand_total:,.6f}
- Bulk demand total: {bulk_demand_total:,.6f}
- Total demand total: {total_demand_total:,.6f}
- Bulk share of total demand: {bulk_share_percentage:.6f}%
- Maximum reconciliation difference: {maximum_reconciliation_difference:.12g}
- Duplicate product-date rows: {duplicate_product_date_rows}
- Negative demand rows: {negative_demand_rows}
- Non-finite demand values: {non_finite_demand_values}

No forecast or fitted model was produced in ND01.
"""

    current_handoff_text = f"""# Current Handoff

- Current completed step: {STEP_ID}
- Status: {STATUS}
- Updated local time: {NOW_LOCAL.isoformat()}
- Workspace root: {MODEL_ROOT}
- Source copied unchanged: {SOURCE_COPY_PATH}
- Source SHA-256: {source_sha256}
- Planned target: NormalDemand
- Bulk included in statistical target: no
- Low-demand products deleted: no
- Models fitted: no
- Next step: ND02

## ND02 objective

Create a leakage-safe normal-demand panel that preserves all products,
calculates product history from prior observations only, identifies established
high/moderate-demand products dynamically, routes other products to fallbacks,
and reports normal-demand coverage before any model is fitted.
"""

    chat_index_text = f"""# Chat and Decision Index

| Date and time | Step | Topic | Status |
|---|---|---|---|
| {NOW_LOCAL.isoformat()} | ND01 | Create normal-demand workspace and copy canonical source | {STATUS} |
"""

    data_dictionary_lines = [
        "# Source Data Dictionary",
        "",
        f"Source file: {SOURCE_COPY_PATH}",
        "",
        "| Column | Data type | Non-null rows | Unique values |",
        "|---|---:|---:|---:|",
    ]
    for column in source_df.columns:
        data_dictionary_lines.append(
            f"| {column} | {source_df[column].dtype} | "
            f"{int(source_df[column].notna().sum()):,} | "
            f"{int(source_df[column].nunique(dropna=True)):,} |"
        )
    data_dictionary_lines.extend(
        [
            "",
            "## Demand-field interpretation",
            "",
            "- NormalDemand: ordinary demand to be modelled in version 2.",
            "- BulkDemand: audit-only demand from confirmed or exceptional bulk transactions.",
            "- TotalDemand: reconciliation field equal to NormalDemand plus BulkDemand.",
            "",
            "No feature engineering was applied in ND01.",
        ]
    )
    data_dictionary_text = "\n".join(data_dictionary_lines) + "\n"

    gitignore_text = """# Eden data and generated model artefacts
01_datasets/00_source_copy/*.csv
01_datasets/01_intermediate/**
01_datasets/02_final/**
03_models/**/*.joblib
03_models/**/*.pkl
03_models/**/*.pickle
04_predictions/**
*.parquet
*.feather

# Jupyter
.ipynb_checkpoints/

# Python
__pycache__/
*.py[cod]

# macOS
.DS_Store
"""

    documentation_files = {
        README_PATH: readme_text,
        AGENTS_PATH: agents_text,
        GITIGNORE_PATH: gitignore_text,
        SOURCE_README_PATH: source_readme_text,
        PROJECT_CONTEXT_PATH: project_context_text,
        WORKFLOW_PATH: workflow_text,
        DECISIONS_PATH: decisions_text,
        FILES_AND_PATHS_PATH: files_and_paths_text,
        METRICS_AND_RESULTS_PATH: metrics_text,
        CURRENT_HANDOFF_PATH: current_handoff_text,
        CHAT_INDEX_PATH: chat_index_text,
        DATA_DICTIONARY_PATH: data_dictionary_text,
    }

    for final_path, text in documentation_files.items():
        staged_path = STAGING_ROOT / final_path.relative_to(MODEL_ROOT)
        write_text(staged_path, text)

    log_text = "\n".join(
        [
            f"Step: {STEP_ID}",
            f"Status: {STATUS}",
            f"Created local: {NOW_LOCAL.isoformat()}",
            f"Created UTC: {NOW_UTC.isoformat()}",
            f"Original source: {SOURCE_PATH}",
            f"Copied source: {SOURCE_COPY_PATH}",
            f"Source SHA256: {source_sha256}",
            f"Rows: {row_count}",
            f"Columns: {column_count}",
            f"Products: {product_count}",
            f"Operating dates: {operating_date_count}",
            f"Normal demand: {normal_demand_total}",
            f"Bulk demand: {bulk_demand_total}",
            f"Total demand: {total_demand_total}",
            "Source modified: False",
            "Dataset transformed: False",
            "Models fitted: False",
            "",
        ]
    )
    staged_log = STAGING_ROOT / LOG_PATH.relative_to(MODEL_ROOT)
    write_text(staged_log, log_text)

    # =========================================================================
    # MANIFEST
    # =========================================================================

    manifest_exclusions = {
        MANIFEST_PATH.relative_to(MODEL_ROOT),
        CHECKPOINT_PATH.relative_to(MODEL_ROOT),
        CHECKPOINT_SHA_PATH.relative_to(MODEL_ROOT),
        LOCK_PATH.relative_to(MODEL_ROOT),
        LOCK_SHA_PATH.relative_to(MODEL_ROOT),
    }

    staged_artifact_files = sorted(
        path
        for path in STAGING_ROOT.rglob("*")
        if path.is_file() and path.relative_to(STAGING_ROOT) not in manifest_exclusions
    )

    manifest = pd.DataFrame(
        [
            {
                "RelativePath": str(path.relative_to(STAGING_ROOT)),
                "Bytes": int(path.stat().st_size),
                "SHA256": sha256_file(path),
            }
            for path in staged_artifact_files
        ]
    ).sort_values("RelativePath").reset_index(drop=True)

    staged_manifest = STAGING_ROOT / MANIFEST_PATH.relative_to(MODEL_ROOT)
    write_csv(staged_manifest, manifest)
    manifest_sha256 = sha256_file(staged_manifest)

    # =========================================================================
    # CHECKPOINT
    # =========================================================================

    checkpoint = {
        "StepID": STEP_ID,
        "Status": STATUS,
        "CreatedUTC": NOW_UTC.isoformat(),
        "CreatedLocal": NOW_LOCAL.isoformat(),
        "WorkspaceRoot": str(MODEL_ROOT),
        "Source": {
            "OriginalPath": str(SOURCE_PATH),
            "CopiedPath": str(SOURCE_COPY_PATH),
            "OriginalSHA256": source_sha256,
            "CopiedSHA256": copied_sha256,
            "ByteIdentical": True,
        },
        "SourceProfile": {
            "Rows": row_count,
            "Columns": column_count,
            "Products": product_count,
            "OperatingDates": operating_date_count,
            "MinimumDate": minimum_date,
            "MaximumDate": maximum_date,
            "NormalDemandTotal": normal_demand_total,
            "BulkDemandTotal": bulk_demand_total,
            "TotalDemandTotal": total_demand_total,
            "BulkSharePercentage": bulk_share_percentage,
        },
        "Manifest": {
            "Path": str(MANIFEST_PATH),
            "SHA256": manifest_sha256,
            "FilesListed": int(len(manifest)),
        },
        "Safety": {
            "OriginalSourceModified": False,
            "CopiedSourceModified": False,
            "RowsRemoved": False,
            "BulkRemoved": False,
            "LowDemandProductsRemoved": False,
            "FeaturesGenerated": False,
            "ModelsFitted": False,
            "ExistingModelLocksModified": False,
        },
        "ReadyForND02": True,
        "NextStep": "ND02",
    }

    staged_checkpoint = STAGING_ROOT / CHECKPOINT_PATH.relative_to(MODEL_ROOT)
    write_json(staged_checkpoint, checkpoint)
    checkpoint_sha256 = sha256_file(staged_checkpoint)

    staged_checkpoint_sha = STAGING_ROOT / CHECKPOINT_SHA_PATH.relative_to(MODEL_ROOT)
    write_text(
        staged_checkpoint_sha,
        f"{checkpoint_sha256}  {CHECKPOINT_PATH.name}\n",
    )

    # =========================================================================
    # LOCK
    # =========================================================================

    lock = {
        "StepID": STEP_ID,
        "Status": STATUS,
        "CreatedUTC": NOW_UTC.isoformat(),
        "WorkspaceRoot": str(MODEL_ROOT),
        "ImmutableSourceCopy": {
            "Path": str(SOURCE_COPY_PATH),
            "SHA256": copied_sha256,
        },
        "SourceSnapshot": {
            "Path": str(SOURCE_SNAPSHOT_PATH),
            "SHA256": sha256_file(staged_source_snapshot),
        },
        "Validation": {
            "Path": str(VALIDATION_PATH),
            "SHA256": sha256_file(staged_validation),
            "AllChecksPassed": bool(validation["Passed"].all()),
        },
        "Manifest": {
            "Path": str(MANIFEST_PATH),
            "SHA256": manifest_sha256,
        },
        "Checkpoint": {
            "Path": str(CHECKPOINT_PATH),
            "SHA256": checkpoint_sha256,
        },
        "SafetyAssertions": checkpoint["Safety"],
        "ReadyForND02": True,
        "NextStep": "ND02",
    }

    staged_lock = STAGING_ROOT / LOCK_PATH.relative_to(MODEL_ROOT)
    write_json(staged_lock, lock)
    lock_sha256 = sha256_file(staged_lock)

    staged_lock_sha = STAGING_ROOT / LOCK_SHA_PATH.relative_to(MODEL_ROOT)
    write_text(staged_lock_sha, f"{lock_sha256}  {LOCK_PATH.name}\n")

    required_staged_files = [
        staged_source_copy,
        staged_source_snapshot,
        staged_validation,
        staged_manifest,
        staged_checkpoint,
        staged_checkpoint_sha,
        staged_lock,
        staged_lock_sha,
        STAGING_ROOT / "README.md",
        STAGING_ROOT / "AGENTS.md",
        STAGING_ROOT / "00_project_memory/CURRENT_HANDOFF.md",
    ]
    missing_staged_files = [path for path in required_staged_files if not path.is_file()]
    if missing_staged_files:
        raise AssertionError(
            "Required staged files are missing:\n"
            + "\n".join(f"- {path}" for path in missing_staged_files)
        )

    if sha256_file(staged_source_copy) != source_sha256:
        raise AssertionError("The staged source-copy hash changed unexpectedly.")

    os.replace(STAGING_ROOT, MODEL_ROOT)

    for protected_path in [
        SOURCE_COPY_PATH,
        SOURCE_SNAPSHOT_PATH,
        VALIDATION_PATH,
        MANIFEST_PATH,
        CHECKPOINT_PATH,
        CHECKPOINT_SHA_PATH,
        LOCK_PATH,
        LOCK_SHA_PATH,
    ]:
        make_read_only(protected_path)

except Exception:
    if STAGING_ROOT.exists():
        shutil.rmtree(STAGING_ROOT)
    raise


# =============================================================================
# FINAL OUTPUT
# =============================================================================

print("=" * 100)
print("EDEN NORMAL-DEMAND MODEL V2 — ND01 COMPLETE")
print("=" * 100)
print(f"Status: {STATUS}")
print(f"Local time: {NOW_LOCAL.isoformat()}")
print(f"Workspace root: {MODEL_ROOT}")

print("\nSOURCE COPY")
print(f"Original source: {SOURCE_PATH}")
print(f"Copied source: {SOURCE_COPY_PATH}")
print(f"SHA-256: {source_sha256}")
print("Byte-identical copy: True")
print("Source modified: False")

print("\nSOURCE PROFILE")
print(f"Rows: {row_count:,}")
print(f"Columns: {column_count:,}")
print(f"Products: {product_count:,}")
print(f"Operating dates: {operating_date_count:,}")
print(f"Date range: {minimum_date} to {maximum_date}")
print(f"Normal demand: {normal_demand_total:,.6f}")
print(f"Bulk demand: {bulk_demand_total:,.6f}")
print(f"Total demand: {total_demand_total:,.6f}")
print(f"Bulk share: {bulk_share_percentage:.6f}%")
print(
    "Maximum reconciliation difference: "
    f"{maximum_reconciliation_difference:.12g}"
)

print("\nAGENT AND PROJECT DOCUMENTATION")
print(f"- AGENTS.md: {AGENTS_PATH}")
print(f"- README.md: {README_PATH}")
print(f"- Current handoff: {CURRENT_HANDOFF_PATH}")
print(f"- Workflow: {WORKFLOW_PATH}")
print(f"- Decisions: {DECISIONS_PATH}")
print(f"- Source dictionary: {DATA_DICTIONARY_PATH}")

print("\nCONTROL FILES")
print(f"- Validation: {VALIDATION_PATH}")
print(f"- Manifest: {MANIFEST_PATH}")
print(f"- Checkpoint: {CHECKPOINT_PATH}")
print(f"- Checkpoint SHA-256: {checkpoint_sha256}")
print(f"- Lock: {LOCK_PATH}")
print(f"- Lock SHA-256: {lock_sha256}")

print("\nSAFETY")
print("- Original dataset overwritten: False")
print("- Source-copy dataset transformed: False")
print("- Bulk demand removed: False")
print("- Low-demand products removed: False")
print("- Features generated: False")
print("- Models fitted: False")
print("- Existing Eden model locks modified: False")

print("\nNEXT STEP")
print(
    "ND02 — create the leakage-safe normal-demand panel, dynamic product "
    "eligibility register, fallback routes and demand-coverage audit."
)
print("=" * 100)

EDEN NORMAL-DEMAND MODEL V2 — ND01 COMPLETE
Status: ND01_SOURCE_WORKSPACE_CREATED_READY_FOR_ND02
Local time: 2026-08-07T21:44:23.660298+01:00
Workspace root: /Users/ryansmac/Desktop/Meng Project/eden_datasets/eden_normal_demand_model_v2

SOURCE COPY
Original source: /Users/ryansmac/Desktop/Meng Project/eden_datasets/UL_EDEN_canonical_product_daily_demand_forecasting_final.csv
Copied source: /Users/ryansmac/Desktop/Meng Project/eden_datasets/eden_normal_demand_model_v2/01_datasets/00_source_copy/UL_EDEN_canonical_product_daily_demand_forecasting_final.csv
SHA-256: f8538b31df4a2751a6b34cbc0a0f82c7d0e1d441480e1c6bdadd8899bca293e2
Byte-identical copy: True
Source modified: False

SOURCE PROFILE
Rows: 25,405
Columns: 38
Products: 227
Operating dates: 245
Date range: 2025-04-01 to 2026-03-30
Normal demand: 114,186.000000
Bulk demand: 1,972.000000
Total demand: 116,158.000000
Bulk share: 1.697688%
Maximum reconciliation difference: 0

AGENT AND PROJECT DOCUMENTATION
- AGENTS.md: /Users/ryansm